# Experimental BCC force constants projected onto the transformed MOGA-PCA manifold

This notebook compares experimental/literature BCC Born--von Kármán force constants against the high-fitness MOGA solution manifold.

The MOGA solution space is represented using the transformed force-constant vector

\[
(\alpha_0,\; u,\; v,\; \alpha_2,\; \beta_2),
\]

where

\[
u = \alpha_1 + 2\beta_1,
\]

and

\[
v = \alpha_1 - \beta_1.
\]

The experimental dataset includes the previously compiled Cr, V, and Fe values in eV/\(\AA^2\), plus BCC Ca values digitized from the screenshot provided in the conversation. The Ca values are converted from N/m to eV/\(\AA^2\) using

\[
1\;\mathrm{N/m} = 0.0624150907\;\mathrm{eV/\AA^2}.
\]

Because the experimental tables usually report interatomic force constants but not the onsite term, \(\alpha_0\) is estimated from the 2NN acoustic-sum-rule condition used in the MOGA workflow:

\[
\alpha_0 + 8\alpha_1 + 2\alpha_2 + 4\beta_2 = 0.
\]

Therefore,

\[
\alpha_0 = -8\alpha_1 - 2\alpha_2 - 4\beta_2.
\]


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

%config InlineBackend.figure_format = "retina"

# -----------------------------------------------------------------------------
# User-adjustable paths
# -----------------------------------------------------------------------------
DATA_DIR = Path("./dataframes/")
EXPERIMENTAL_CSV = Path("./bcc_experimental_force_constants_alpha_beta.csv")

DATAFRAME_FILES = {
    "dataframe000013": DATA_DIR / "dataframe000013.pkl",
    "dataframe000014": DATA_DIR / "dataframe000014.pkl",
    "dataframe000015": DATA_DIR / "dataframe000015.pkl",
    "dataframe000016": DATA_DIR / "dataframe000016.pkl",
}

TOP_K = 5
RANK_BY = "fitness_norm"

OUTPUT_DIR = Path("experimental_force_constants_pca_overlay")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAVE_FORMATS = ("pdf", "png")
DPI = 300

FIGSIZE = (6.4, 5.4)
MARKER_SIZE = 58
MARKER_ALPHA = 0.78
EDGE_WIDTH = 0.3

NM_TO_EV_A2 = 0.06241509074460765


In [ ]:
def find_first_existing(candidates, columns):
    for col in candidates:
        if col in columns:
            return col
    return None


def save_figure(fig, stem, output_dir=OUTPUT_DIR):
    for ext in SAVE_FORMATS:
        path = output_dir / f"{stem}.{ext}"
        fig.savefig(path, dpi=DPI, bbox_inches="tight")
        print(f"Saved {path}")


def apply_publication_axes(ax):
    ax.tick_params(direction="in", top=True, right=True, labelsize=12)
    for spine in ax.spines.values():
        spine.set_linewidth(1.1)


def display_label(col):
    label_map = {
        "alpha0": r"$\alpha_0$ (eV/$\AA^2$)",
        "alpha1": r"$\alpha_1$ (eV/$\AA^2$)",
        "beta1": r"$\beta_1$ (eV/$\AA^2$)",
        "alpha2": r"$\alpha_2$ (eV/$\AA^2$)",
        "beta2": r"$\beta_2$ (eV/$\AA^2$)",
        "alpha1_plus_2beta1": r"$\alpha_1 + 2\beta_1$ (eV/$\AA^2$)",
        "alpha1_minus_beta1": r"$\alpha_1 - \beta_1$ (eV/$\AA^2$)",
    }
    return label_map.get(col, col)


def add_derived_force_constants(df):
    df = df.copy()
    df["alpha1_plus_2beta1"] = df["alpha1"] + 2.0 * df["beta1"]
    df["alpha1_minus_beta1"] = df["alpha1"] - df["beta1"]
    df["r_alpha2"] = np.abs(df["alpha2"]) / (np.abs(df["alpha2"]) + np.abs(df["beta2"]) + 1e-12)
    return df


def assign_family(row, threshold=0.10):
    a2 = abs(row["alpha2"])
    b2 = abs(row["beta2"])
    if a2 > b2 * (1.0 + threshold):
        return r"$\alpha_2$-dominated"
    if b2 > a2 * (1.0 + threshold):
        return r"$\beta_2$-dominated"
    if row["alpha2"] >= row["beta2"]:
        return r"$\alpha_2$ larger"
    return r"$\beta_2$ larger"


## Load the MOGA top-5 overall solutions

In [ ]:
frames = []
for label, path in DATAFRAME_FILES.items():
    if not path.exists():
        raise FileNotFoundError(f"Could not find {path}. Update DATA_DIR or DATAFRAME_FILES.")
    tmp = pd.read_pickle(path).copy()
    tmp["dataset"] = label
    print(f"{label}: {tmp.shape[0]:,} rows, {tmp.shape[1]:,} columns")
    frames.append(tmp)

df_all = pd.concat(frames, ignore_index=True)
print(f"\nAggregated dataframe before filtering: {df_all.shape[0]:,} rows")

mass_col = find_first_existing(["mass", "m"], df_all.columns)
alat_col = find_first_existing(["a_val", "alat", "a_latt", "lattice_parameter"], df_all.columns)
rank_col = find_first_existing([RANK_BY, "fitness_norm", "fnorm"], df_all.columns)

col_map = {
    "alpha0": find_first_existing(["alpha0", "alpha_0"], df_all.columns),
    "alpha1": find_first_existing(["alpha1", "alpha_1"], df_all.columns),
    "beta1":  find_first_existing(["beta1", "beta_1"], df_all.columns),
    "alpha2": find_first_existing(["alpha2", "alpha_2"], df_all.columns),
    "beta2":  find_first_existing(["beta2", "beta_2"], df_all.columns),
}

missing = [k for k, v in col_map.items() if v is None]
if missing:
    raise ValueError(f"Missing required force-constant columns: {missing}")
if mass_col is None or alat_col is None or rank_col is None:
    raise ValueError("Could not identify mass, lattice parameter, or ranking column.")

# Standardize column names used below.
df_all = df_all.rename(columns={v: k for k, v in col_map.items()})
if mass_col != "mass":
    df_all = df_all.rename(columns={mass_col: "mass"})
if alat_col != "a_val":
    df_all = df_all.rename(columns={alat_col: "a_val"})

sort_ascending = False
sort_cols = ["dataset", "mass", "a_val", rank_col]
df_top = (
    df_all.sort_values(sort_cols, ascending=[True, True, True, sort_ascending])
          .groupby(["dataset", "mass", "a_val"], as_index=False, group_keys=False)
          .head(TOP_K)
          .reset_index(drop=True)
)

df_top = add_derived_force_constants(df_top)
df_top["family"] = df_top.apply(assign_family, axis=1)

print(f"Top-{TOP_K} selected dataframe: {df_top.shape[0]:,} rows")
df_top[["dataset", "mass", "a_val", "alpha0", "alpha1", "beta1", "alpha2", "beta2", "alpha1_plus_2beta1", "alpha1_minus_beta1", "r_alpha2"]].head()


## Fit PCA using the transformed MOGA force-constant basis

In [ ]:
FEATURE_COLS = [
    "alpha0",
    "alpha1_plus_2beta1",
    "alpha1_minus_beta1",
    "alpha2",
    "beta2",
]

X = df_top[FEATURE_COLS].astype(float).to_numpy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=5)
scores = pca.fit_transform(X_scaled)

for i in range(scores.shape[1]):
    df_top[f"PC{i+1}"] = scores[:, i]

explained = pca.explained_variance_ratio_
print("Explained variance ratio:")
for i, val in enumerate(explained, start=1):
    print(f"PC{i}: {val:.4f} ({100*val:.1f}%)")

loadings = pd.DataFrame(
    pca.components_.T,
    index=[display_label(c) for c in FEATURE_COLS],
    columns=[f"PC{i}" for i in range(1, 6)],
)
loadings


## Build the experimental force-constant table

The Cr, V, and Fe values are read from the compiled CSV. The BCC Ca values from the screenshot are added manually below.

The Ca screenshot reports force constants in N/m with the convention

- `1xx` \(\rightarrow -\alpha_1\)
- `1xy` \(\rightarrow -\beta_1\)
- `2xx` \(\rightarrow -\alpha_2\)
- `2xy` \(\rightarrow -\beta_2\)

under the sign convention used for the previous Fe/V/Cr table.


In [ ]:
def load_compiled_experimental_csv(path):
    if not path.exists():
        raise FileNotFoundError(
            f"Could not find {path}. Place bcc_experimental_force_constants_alpha_beta.csv "
            "next to this notebook or update EXPERIMENTAL_CSV."
        )
    exp = pd.read_csv(path).copy()

    # Accept either clean or unit-explicit column names.
    rename = {
        "alpha1_eV_A2": "alpha1",
        "beta1_eV_A2": "beta1",
        "alpha2_eV_A2": "alpha2",
        "beta2_eV_A2": "beta2",
        "temperature_K": "temperature_K",
        "element": "element",
    }
    exp = exp.rename(columns={k: v for k, v in rename.items() if k in exp.columns})

    required = ["element", "temperature_K", "alpha1", "beta1", "alpha2", "beta2"]
    missing = [c for c in required if c not in exp.columns]
    if missing:
        raise ValueError(f"Experimental CSV is missing required columns: {missing}")

    exp = exp[required].copy()
    exp["source"] = "compiled_csv"
    return exp


exp_csv = load_compiled_experimental_csv(EXPERIMENTAL_CSV)

# BCC Ca values from the screenshot. Original units are N/m.
ca_nm = pd.DataFrame([
    {"element": "Ca", "temperature_K": 726, "one_xx_N_m": 4.014, "one_xy_N_m": 3.503, "two_xx_N_m": 1.443, "two_xy_N_m": -1.182},
    {"element": "Ca", "temperature_K": 750, "one_xx_N_m": 3.936, "one_xy_N_m": 3.582, "two_xx_N_m": 1.441, "two_xy_N_m": -0.926},
])

ca_exp = pd.DataFrame({
    "element": ca_nm["element"],
    "temperature_K": ca_nm["temperature_K"],
    "alpha1": -ca_nm["one_xx_N_m"] * NM_TO_EV_A2,
    "beta1":  -ca_nm["one_xy_N_m"] * NM_TO_EV_A2,
    "alpha2": -ca_nm["two_xx_N_m"] * NM_TO_EV_A2,
    "beta2":  -ca_nm["two_xy_N_m"] * NM_TO_EV_A2,
    "source": "Ca_screenshot_converted_from_N_per_m",
})

exp_all = pd.concat([exp_csv, ca_exp], ignore_index=True)

# Estimate alpha0 using the same acoustic-sum-rule combination used in the MOGA fitness.
exp_all["alpha0"] = -8.0 * exp_all["alpha1"] - 2.0 * exp_all["alpha2"] - 4.0 * exp_all["beta2"]
exp_all = add_derived_force_constants(exp_all)
exp_all["family"] = exp_all.apply(assign_family, axis=1)

exp_all = exp_all.sort_values(["element", "temperature_K"]).reset_index(drop=True)

cols_to_show = [
    "element", "temperature_K", "alpha0", "alpha1", "beta1", "alpha1_plus_2beta1", "alpha1_minus_beta1", "alpha2", "beta2", "r_alpha2", "family", "source"
]

pd.set_option("display.max_rows", 200)
exp_all[cols_to_show]


## Project the experimental force constants into the MOGA PCA space

In [ ]:
X_exp = exp_all[FEATURE_COLS].astype(float).to_numpy()
X_exp_scaled = scaler.transform(X_exp)
exp_scores = pca.transform(X_exp_scaled)

for i in range(exp_scores.shape[1]):
    exp_all[f"PC{i+1}"] = exp_scores[:, i]

exp_all.to_csv(OUTPUT_DIR / "experimental_force_constants_with_transformed_coordinates_and_pca_scores.csv", index=False)
df_top.to_csv(OUTPUT_DIR / "moga_top5_transformed_pca_scores.csv", index=False)
loadings.to_csv(OUTPUT_DIR / "pca_loadings_transformed_basis.csv")

print("Experimental PCA scores:")
exp_all[["element", "temperature_K", "PC1", "PC2", "PC3", "alpha1_plus_2beta1", "alpha1_minus_beta1", "r_alpha2"]]


## PCA overlay plots

The gray points are the high-fitness MOGA solutions used to fit PCA. The colored markers are the experimental materials projected into the same standardized PCA basis.


In [ ]:
element_markers = {
    "Fe": "o",
    "Cr": "s",
    "V": "^",
    "Ca": "D",
}

element_colors = {
    "Fe": "tab:red",
    "Cr": "tab:purple",
    "V": "tab:green",
    "Ca": "tab:blue",
}

fig, ax = plt.subplots(figsize=FIGSIZE)

ax.scatter(
    df_top["PC1"], df_top["PC2"],
    s=30, alpha=0.22, c="0.65",
    edgecolors="none", label="MOGA top-5 solutions",
)

for element, sub in exp_all.groupby("element"):
    sub = sub.sort_values("temperature_K")
    ax.plot(
        sub["PC1"], sub["PC2"],
        linewidth=1.5, alpha=0.8,
        color=element_colors.get(element, None),
        zorder=3,
    )
    ax.scatter(
        sub["PC1"], sub["PC2"],
        s=105,
        marker=element_markers.get(element, "o"),
        color=element_colors.get(element, None),
        edgecolors="black", linewidths=0.8,
        label=element,
        zorder=4,
    )

ax.set_xlabel(f"PC1 ({explained[0]*100:.1f}%)", fontsize=14)
ax.set_ylabel(f"PC2 ({explained[1]*100:.1f}%)", fontsize=14)
ax.set_title("Experimental BCC force constants projected onto MOGA PCA", fontsize=15, pad=10)
apply_publication_axes(ax)
ax.legend(frameon=False, fontsize=9, loc="best")
fig.tight_layout()
save_figure(fig, "pca_overlay_experimental_by_element")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=FIGSIZE)

ax.scatter(
    df_top["PC1"], df_top["PC2"],
    s=30, alpha=0.18, c="0.65",
    edgecolors="none", label="MOGA top-5 solutions",
)

sc = ax.scatter(
    exp_all["PC1"], exp_all["PC2"],
    c=exp_all["temperature_K"].astype(float),
    cmap="viridis",
    s=115,
    edgecolors="black", linewidths=0.8,
    zorder=4,
)

for _, row in exp_all.iterrows():
    label = f"{row['element']} {int(row['temperature_K'])} K"
    ax.annotate(label, (row["PC1"], row["PC2"]), xytext=(4, 4), textcoords="offset points", fontsize=8)

cbar = fig.colorbar(sc, ax=ax)
cbar.set_label(r"$T$ (K)", fontsize=13)
cbar.ax.tick_params(labelsize=11)

ax.set_xlabel(f"PC1 ({explained[0]*100:.1f}%)", fontsize=14)
ax.set_ylabel(f"PC2 ({explained[1]*100:.1f}%)", fontsize=14)
ax.set_title("Experimental BCC force constants colored by temperature", fontsize=15, pad=10)
apply_publication_axes(ax)
fig.tight_layout()
save_figure(fig, "pca_overlay_experimental_colored_by_temperature")
plt.show()


## Transformed first-neighbor coordinates

This plot directly compares the experimental values in the physically motivated first-neighbor combinations

\[
u = \alpha_1 + 2\beta_1,
\]

and

\[
v = \alpha_1 - \beta_1.
\]


In [ ]:
fig, ax = plt.subplots(figsize=FIGSIZE)

ax.scatter(
    df_top["alpha1_plus_2beta1"], df_top["alpha1_minus_beta1"],
    s=30, alpha=0.18, c="0.65", edgecolors="none", label="MOGA top-5 solutions",
)

for element, sub in exp_all.groupby("element"):
    sub = sub.sort_values("temperature_K")
    ax.plot(
        sub["alpha1_plus_2beta1"], sub["alpha1_minus_beta1"],
        linewidth=1.5, alpha=0.8,
        color=element_colors.get(element, None),
        zorder=3,
    )
    ax.scatter(
        sub["alpha1_plus_2beta1"], sub["alpha1_minus_beta1"],
        s=105,
        marker=element_markers.get(element, "o"),
        color=element_colors.get(element, None),
        edgecolors="black", linewidths=0.8,
        label=element,
        zorder=4,
    )

ax.axhline(0.0, color="0.8", linewidth=1.0)
ax.axvline(0.0, color="0.8", linewidth=1.0)
ax.set_xlabel(r"$\alpha_1 + 2\beta_1$ (eV/$\AA^2$)", fontsize=14)
ax.set_ylabel(r"$\alpha_1 - \beta_1$ (eV/$\AA^2$)", fontsize=14)
ax.set_title("Experimental first-neighbor transformed coordinates", fontsize=15, pad=10)
apply_publication_axes(ax)
ax.legend(frameon=False, fontsize=9, loc="best")
fig.tight_layout()
save_figure(fig, "experimental_transformed_first_neighbor_coordinates")
plt.show()


## Second-neighbor balance

This plot compares \(\alpha_2\), \(\beta_2\), and the ratio

\[
r_{\alpha_2} = \frac{|\alpha_2|}{|\alpha_2| + |\beta_2|}.
\]


In [ ]:
fig, ax = plt.subplots(figsize=FIGSIZE)

ax.scatter(
    df_top["alpha2"], df_top["beta2"],
    c=df_top["r_alpha2"], cmap="coolwarm", vmin=0, vmax=1,
    s=34, alpha=0.18, edgecolors="none", label="MOGA top-5 solutions",
)

for element, sub in exp_all.groupby("element"):
    sub = sub.sort_values("temperature_K")
    ax.plot(
        sub["alpha2"], sub["beta2"],
        linewidth=1.5, alpha=0.8,
        color=element_colors.get(element, None),
        zorder=3,
    )
    ax.scatter(
        sub["alpha2"], sub["beta2"],
        s=105,
        marker=element_markers.get(element, "o"),
        color=element_colors.get(element, None),
        edgecolors="black", linewidths=0.8,
        label=element,
        zorder=4,
    )

ax.axhline(0.0, color="0.8", linewidth=1.0)
ax.axvline(0.0, color="0.8", linewidth=1.0)
ax.set_xlabel(r"$\alpha_2$ (eV/$\AA^2$)", fontsize=14)
ax.set_ylabel(r"$\beta_2$ (eV/$\AA^2$)", fontsize=14)
ax.set_title("Experimental second-neighbor force constants", fontsize=15, pad=10)
apply_publication_axes(ax)
ax.legend(frameon=False, fontsize=9, loc="best")
fig.tight_layout()
save_figure(fig, "experimental_second_neighbor_force_constants")
plt.show()


## Print the final experimental table

In [ ]:
print(exp_all[cols_to_show].to_string(index=False, float_format=lambda x: f"{x: .6f}"))
